In [6]:
import pickle
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from ta.momentum import RSIIndicator, ROCIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands, AverageTrueRange

In [3]:
with open("model_cat.pkl", "rb") as file:
    model_cat = pickle.load(file)

print("CatBoost model loaded successfully!")

CatBoost model loaded successfully!


In [4]:
# Load the original dataset
data = pd.read_csv("merged_data.csv")

# Convert Date to datetime
data["Date"] = pd.to_datetime(data["Date"])

print("Dataset shape:", data.shape)
print("Date range:", data["Date"].min(), "to", data["Date"].max())

Dataset shape: (1381571, 7)
Date range: 2000-01-01 00:00:00 to 2025-02-26 00:00:00


## **Feature Engineering**

In [ ]:
# Load original dataset
data = pd.read_csv("merged_data.csv")

# Convert Date
data["Date"] = pd.to_datetime(data["Date"])

# Sort exactly by Scrip and Date
data = data.sort_values(
    ["Scrip", "Date"]
).reset_index(drop=True)


# Feature Engineering
def add_features(group):

    close = group["Close"]
    high = group["High"]
    low = group["Low"]
    volume = group["Volume"]

    # Return
    group["Return"] = close.pct_change()

    # Lagged Returns
    group["Return_Lag1"] = group["Return"].shift(1)
    group["Return_Lag2"] = group["Return"].shift(2)
    group["Return_Lag5"] = group["Return"].shift(5)

    # Volatility
    group["Volatility_5"] = group["Return"].rolling(5).std()
    group["Volatility_20"] = group["Return"].rolling(20).std()

    # Price Range
    group["Price_Range"] = (high - low) / close

    # Volume Change
    group["Volume_Change"] = volume.pct_change()

    # Volume SMA
    group["Volume_SMA_20"] = volume.rolling(20).mean()

    # RSI
    rsi = RSIIndicator(close=close)
    group["RSI"] = rsi.rsi()

    # MACD
    macd = MACD(close=close)

    group["MACD"] = macd.macd()
    group["MACD_Signal"] = macd.macd_signal()
    group["MACD_Hist"] = macd.macd_diff()

    # Bollinger Band
    bb = BollingerBands(close=close)

    group["BB_High"] = bb.bollinger_hband()

    # ATR
    atr = AverageTrueRange(
        high=high,
        low=low,
        close=close
    )

    group["ATR"] = atr.average_true_range()

    # ROC
    roc = ROCIndicator(close=close)

    group["ROC"] = roc.roc()

    return group


# Apply feature engineering separately for each stock
data = data.groupby(
    "Scrip",
    group_keys=False
).apply(add_features)

data = data.reset_index(drop=True)

print("Feature engineering completed!")
print("Shape:", data.shape)

IndexError: index 13 is out of bounds for axis 0 with size 4